# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant-python/) library.

### Dataset Source

This dataset is described in a Croissant schema, accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

*Citation:* Kamadi, V, Chimoita, EL, Wahome, RG and Odhong, C (2026). "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya, Frontiers."

In [ ]:
# Install mlcroissant if not already installed!pip install mlcroissant

## 1. Data Loading

Let's load the dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata (schema)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Dataset Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview

Let's review the available record sets (tables), fields (columns/attributes), and their unique `@id` identifiers. This helps ensure we're referencing data elements correctly for extraction and analysis.

**Note:** The `@id` fields are crucial for uniquely identifying each record set and field throughout Croissant descriptions and in mlcroissant.

In [ ]:
# List all available record sets by their `@id` and name if present
def get_record_set_overview(ds):
    print("Available record sets:")
    for record_set in ds.record_sets:
        rec_id = record_set['@id']
        rec_name = record_set.get('name', '<unnamed>')
        print(f"  - @id: {rec_id}, name: {rec_name}")
        # Print a preview of field @ids for this record set
        if 'field' in record_set:
            print("      fields:")
            for field in record_set['field']:
                if isinstance(field, dict):
                    print(f"        - {field.get('@id', '<unnamed field>')}")
                else:
                    print(f"        - {field}")
        print("")

get_record_set_overview(dataset)

## 3. Data Extraction

We'll now load the available record sets as DataFrames to explore and analyze them.

Refer to the `@id` identifiers for each record set and their fields, as shown above. If there are multiple record sets, we'll load each into a separate DataFrame. If none, this section will note as such.

In [ ]:
# Identify all record sets from the dataset metadata
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
if not record_set_ids:
    print("No record sets defined in the Croissant metadata.")
else:
    print(f"Found {len(record_set_ids)} record set(s):")
    for rsid in record_set_ids:
        print(f"  - {rsid}")

# Extract all records for each record set
dataframes = {}
for rsid in record_set_ids:
    try:
        rows = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(rows)
        dataframes[rsid] = df
        print(f"{rsid}: loaded {len(df)} records. Columns: {list(df.columns)}")
    except Exception as ex:
        print(f"Error loading records for {rsid}: {ex}")

# As an example, preview the first 5 rows of the first available record set, if present
if record_set_ids:
    example_rsid = record_set_ids[0]
    print(f"\nPreview of '{example_rsid}':")
    display(dataframes[example_rsid].head())

## 4. Exploratory Data Analysis (EDA)

Let's conduct some common EDA operations. We'll demonstrate filtering, normalizing, and grouping, referencing fields by their `@id`. We'll focus on the first record set found (if available) as an example.

In [ ]:
import numpy as np
import warnings

if not record_set_ids:
    print("No record sets available for EDA.")
else:
    rsid = record_set_ids[0]
    df = dataframes[rsid]
    print(f"Working with record set: {rsid}")
    print(f"Available columns (@id): {list(df.columns)}\n")
    # Attempt to identify a likely numeric field by type or name
    # Fallback to first numeric-looking column
    numeric_field = None
    for col in df.columns:
        # Try to detect numeric columns by examining dtype or common names
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    if not numeric_field:
        # Try columns with numeric in field @id
        for col in df.columns:
            if 'log_likelihood' in col or 'coefficient' in col or 'std_err' in col or 'p_value' in col or 'num' in col:
                numeric_field = col
                break
    if not numeric_field:
        print("Could not automatically find a numeric field for EDA. Please set manually.")
    else:
        print(f"Numeric field chosen for EDA: {numeric_field}\n")

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            # Filter records where numeric_field > threshold
            threshold = df[numeric_field].mean(skipna=True)
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
            display(filtered_df.head())

            # Normalize the field
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"\nNormalized values of {numeric_field}:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try grouping by a categorical field (e.g., 'ward', 'gender', etc.)
            group_field = None
            for col in df.columns:
                if col.lower() in ('ward', 'gender', 'county', 'category'):
                    group_field = col
                    break
            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()  # Mean per group
                print(f"\nMean {numeric_field} grouped by {group_field}:")
                display(grouped_df.head())
            else:
                print("No obvious group/categorical field found to demonstrate grouping.")

## 5. Visualization

Let's visualize the distribution of a numeric field and, if possible, compare by a grouping variable. This helps us see the patterns or outliers in the data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids or not numeric_field:
    print("No numeric data available for plotting.")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, ax=ax, color='skyblue')
    ax.set_title(f"Distribution of {numeric_field}")
    ax.set_xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    # Boxplot by group if grouping field was found
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to reference data by unique `@id` using `mlcroissant`.
- We loaded dataset metadata, explored record sets and fields, and analyzed numeric data with filtering, normalization, and grouping.
- Visualizations revealed data distributions and potential group differences.

**Recommendations:** For deeper analysis or model-building, further investigation into detailed field meanings (from the data dictionary, if available) is advised. Make sure to consult the dataset's documentation for context, especially for sensitive columns or known biases described in the metadata.

For more information, see the [FAIR² dataset page](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) or refer to mlcroissant's official [documentation](https://mlcommons.github.io/croissant-python/).